# CoreMs to MetaboDirect Transformation Pipeline

This notebook loads CoreMs output CSV files from `CoreMs_input/`, maps their column
schema to the format expected by MetaboDirect's transformation engine, and computes
biochemical transformations between mass peaks.

**Outputs** (written to `output/`):
- `transf_by_sample/transformations_<SampleID>.csv` - matched transformation pairs per sample
- `transf_by_sample/counts_<SampleID>.csv` - transformation counts per sample
- `Transformations_summary_counts.csv` - merged counts across all samples
- `Transformations_summary_all.csv` - merged transformation pairs across all samples
- `node_table.csv` - compound metadata for network nodes

In [6]:
import os
import re
import glob
import sys
import datetime
import pandas as pd
import numpy as np

In [7]:
# Mock heavy optional deps used only by Cytoscape network creation;
# this lets us import the transformation functions without needing
# py4cytoscape or a running Cytoscape instance.
from unittest.mock import MagicMock

for _mod in ['seaborn', 'py4cytoscape']:
    if _mod not in sys.modules:
        sys.modules[_mod] = MagicMock()

# Ensure the metabodirect package is importable from the project root
sys.path.insert(0, os.getcwd())

from metabodirect.transformations import (
    get_keys,
    calculate_transformations,
    summarize_transformations,
    get_node_table,
)
print('Transformation functions imported successfully.')

Transformation functions imported successfully.


In [8]:
# -- Paths ------------------------------------------------------------------
COREMS_FOLDER = os.path.join(os.getcwd(), 'CoreMs_input')
KEY_FILE      = os.path.join(os.getcwd(), 'metabodirect', 'data', 'Transformation_Database_07-2020.csv')
OUTPUT_DIR    = os.path.join(os.getcwd(), 'output')
TRANSF_DIR    = os.path.join(OUTPUT_DIR, 'transf_by_sample')

# Transformation mass-error tolerance (Da)
ERR_THRESH = 0.000010

print('CoreMs input :', COREMS_FOLDER)
print('Key file     :', KEY_FILE)
print('Output dir   :', OUTPUT_DIR)

CoreMs input : c:\Users\gara009\OneDrive - PNNL\Documents\GitHub\RC2_Transformations\CoreMs_input
Key file     : c:\Users\gara009\OneDrive - PNNL\Documents\GitHub\RC2_Transformations\metabodirect\data\Transformation_Database_07-2020.csv
Output dir   : c:\Users\gara009\OneDrive - PNNL\Documents\GitHub\RC2_Transformations\output


In [9]:
def load_corems_files(corems_folder):
    '''Load all *.corems.csv files and return a long-format DataFrame.

    CoreMs column mapping to MetaboDirect schema
    --------------------------------------------
    Calibrated_Mass   -> Mass
    Peak_Height       -> NormIntensity
    OtoC_ratio        -> OC
    HtoC_ratio        -> HC
    Heteroatom_Class  -> Class
    Molecular_Formula -> MolecularFormula (spaces removed)
    filename stem     -> SampleID

    Derived columns (same formulas as metabodirect/preprocessing.py)
    ----------------------------------------------------------------
    El_comp : elemental composition without stoichiometry (e.g. CHO)
    NOSC    : nominal oxidation state of carbon
    GFE     : Gibbs free energy of C oxidation
    '''
    files = sorted(glob.glob(os.path.join(corems_folder, '*.corems.csv')))
    if not files:
        raise FileNotFoundError(f'No .corems.csv files found in: {corems_folder}')

    frames = []
    for fpath in files:
        sample_id = os.path.basename(fpath).replace('.corems.csv', '')
        raw = pd.read_csv(fpath, na_values=['-9999', -9999])

        # Keep only monoisotopic peaks with a valid formula assignment
        raw = raw[raw['Is_Isotopologue'] == 0].copy()
        raw = raw[raw['Molecular_Formula'].notna()].copy()
        raw = raw[raw['Molecular_Formula'].str.strip() != ''].copy()

        # Drop peaks with missing or zero intensity
        raw = raw[raw['Peak_Height'].notna() & (raw['Peak_Height'] > 0)].copy()

        if raw.empty:
            print(f'  WARNING: no valid peaks in {os.path.basename(fpath)}, skipping.')
            continue

        # Fill missing element counts with 0
        for el in ['C', 'H', 'O', 'N', 'P', 'S']:
            raw[el] = pd.to_numeric(raw[el], errors='coerce').fillna(0)

        # Column mapping
        raw['Mass']             = raw['Calibrated_Mass']
        raw['NormIntensity']    = raw['Peak_Height']
        raw['OC']               = raw['OtoC_ratio']
        raw['HC']               = raw['HtoC_ratio']
        raw['Class']            = raw['Heteroatom_Class']
        raw['MolecularFormula'] = raw['Molecular_Formula'].str.replace(' ', '', regex=False)
        raw['SampleID']         = sample_id

        # Derived indices
        raw['El_comp'] = raw['MolecularFormula'].apply(
            lambda mf: re.sub(r'\d', '', mf)
        )
        c = raw['C'].replace(0, np.nan)  # Avoid division by zero for C=0 rows
        raw['NOSC'] = (
            -((4 * raw['C'] + raw['H'] - 3 * raw['N']
               - 2 * raw['O'] + 5 * raw['P'] - 2 * raw['S']) / c)
            + 4
        )
        raw['GFE'] = -(28.5 * raw['NOSC']) + 60.3

        frames.append(raw)
        print(f'  {sample_id}: {len(raw):,} peaks')

    if not frames:
        raise RuntimeError('No valid data loaded from any CoreMs file.')

    combined = pd.concat(frames, ignore_index=True)
    print(f'\nTotal: {len(combined):,} peaks across {len(frames)} samples.')
    return combined


In [10]:
print('Loading CoreMs files...')
df = load_corems_files(COREMS_FOLDER)

cols = ['Mass', 'NormIntensity', 'SampleID', 'C', 'H', 'O',
        'OC', 'HC', 'NOSC', 'GFE', 'Class', 'MolecularFormula', 'El_comp']
df[cols].head(8)

Loading CoreMs files...
  RC2_0001_ICR-1_p08: 4,182 peaks
  RC2_0001_ICR-2_p08: 4,373 peaks
  RC2_0001_ICR-3_p08: 4,130 peaks
  RC2_0002_ICR-1_p08: 4,475 peaks
  RC2_0002_ICR-2_p08: 4,428 peaks
  RC2_0002_ICR-3_p08: 4,249 peaks
  RC2_0003_ICR-1_p08: 4,317 peaks
  RC2_0003_ICR-2_p08: 3,702 peaks
  RC2_0003_ICR-3_p08: 4,202 peaks
  RC2_0004_ICR-1_p08: 4,434 peaks
  RC2_0004_ICR-2_p08: 4,462 peaks
  RC2_0004_ICR-3_p08: 4,239 peaks
  RC2_0005_ICR-1_p08: 4,574 peaks
  RC2_0005_ICR-2_p08: 4,618 peaks
  RC2_0005_ICR-3_p08: 4,123 peaks
  RC2_0006_ICR-1_p08: 4,552 peaks
  RC2_0006_ICR-2_p08: 4,721 peaks
  RC2_0006_ICR-3_p08: 4,517 peaks
  RC2_0007_ICR-1_p08: 4,112 peaks
  RC2_0007_ICR-2_p08: 4,226 peaks
  RC2_0007_ICR-3_p08: 3,873 peaks
  RC2_0008_ICR-1_p08: 4,422 peaks
  RC2_0008_ICR-2_p08: 4,237 peaks
  RC2_0008_ICR-3_p08: 4,429 peaks
  RC2_0009_ICR-1_p08: 4,305 peaks
  RC2_0009_ICR-2_p08: 4,500 peaks
  RC2_0009_ICR-3_p08: 4,461 peaks
  RC2_0010_ICR-1_p08: 4,398 peaks
  RC2_0010_ICR-2_p08: 4,

,Mass,NormIntensity,SampleID,C,H,O,OC,HC,NOSC,GFE,Class,MolecularFormula,El_comp
0,202.970305,2238973.0,RC2_0001_ICR-1_p08,4,4,2,0.500000,1.000000,4.000000,-53.700000,N4 S2 O2,C4H4O2S2N4,CHOSN
1,238.963145,3304118.5,RC2_0001_ICR-1_p08,13,4,1,0.076923,0.307692,0.153846,55.915385,S2 O1,C13H4O1S2,CHOS
2,247.097696,2525221.5,RC2_0001_ICR-1_p08,14,16,4,0.285714,1.142857,-0.571429,76.585714,O4,C14H16O4,CHO
3,252.978797,7009289.0,RC2_0001_ICR-1_p08,14,6,1,0.071429,0.428571,0.000000,60.300000,S2 O1,C14H6O1S2,CHOS
4,255.233050,1404377.9,RC2_0001_ICR-1_p08,16,32,2,0.125000,2.000000,-1.750000,110.175000,O2,C16H32O2,CHO
5,257.139562,2766102.3,RC2_0001_ICR-1_p08,13,22,5,0.384615,1.692308,-0.923077,86.607692,O5,C13H22O5,CHO
6,261.040506,1550265.5,RC2_0001_ICR-1_p08,13,10,6,0.461538,0.769231,0.153846,55.915385,O6,C13H10O6,CHO
7,261.076937,2858441.5,RC2_0001_ICR-1_p08,14,14,5,0.357143,1.000000,-0.285714,68.442857,O5,C14H14O5,CHO


In [11]:
keys = get_keys(KEY_FILE)
print(f'Loaded {len(keys)} transformation keys.')
pd.read_csv(KEY_FILE).head(10)

Loaded 1255 transformation keys.


,Formula,mf,Group,Transformation
0,CH4_O,0.036380,Other,random
1,NH_CH2,0.995250,Other,random
2,2ndIP,1.997000,Other,random
3,Na_H,21.981940,Other,random
4,CH3COO-,59.013305,Other,random
5,PO43,94.953423,Other,random
6,SO42,95.951732,Other,random
7,NO3,61.987819,Other,random
8,NH4,18.034374,Other,random
9,H2S,33.987722,Other,random


In [12]:
os.makedirs(TRANSF_DIR, exist_ok=True)
print('Output directories created:')
print(' ', OUTPUT_DIR)
print(' ', TRANSF_DIR)

Output directories created:
  c:\Users\gara009\OneDrive - PNNL\Documents\GitHub\RC2_Transformations\output
  c:\Users\gara009\OneDrive - PNNL\Documents\GitHub\RC2_Transformations\output\transf_by_sample


In [13]:
print('Calculating transformations (this may take several minutes per sample)...\n')
calculate_transformations(df, keys, TRANSF_DIR, err_thresh=ERR_THRESH)

Calculating transformations (this may take several minutes per sample)...

[2026-07-01 17:21:56 PM]1\576	RC2_0001_ICR-1_p08
		Total m/z values 4182
   Saving results
[2026-07-01 17:22:04 PM]2\576	RC2_0001_ICR-2_p08
		Total m/z values 4373
   Saving results
[2026-07-01 17:22:14 PM]3\576	RC2_0001_ICR-3_p08
		Total m/z values 4130
   Saving results
[2026-07-01 17:22:22 PM]4\576	RC2_0002_ICR-1_p08
		Total m/z values 4475
   Saving results
[2026-07-01 17:22:31 PM]5\576	RC2_0002_ICR-2_p08
		Total m/z values 4428
   Saving results
[2026-07-01 17:22:41 PM]6\576	RC2_0002_ICR-3_p08
		Total m/z values 4249
   Saving results
[2026-07-01 17:22:49 PM]7\576	RC2_0003_ICR-1_p08
		Total m/z values 4317
   Saving results
[2026-07-01 17:22:58 PM]8\576	RC2_0003_ICR-2_p08
		Total m/z values 3702
   Saving results
[2026-07-01 17:23:05 PM]9\576	RC2_0003_ICR-3_p08
		Total m/z values 4202
   Saving results
[2026-07-01 17:23:13 PM]10\576	RC2_0004_ICR-1_p08
		Total m/z values 4434
   Saving results
[2026-07-01 17

In [14]:
summarize_transformations(OUTPUT_DIR)
print('\nSummary files written to:', OUTPUT_DIR)


Summary files written to: c:\Users\gara009\OneDrive - PNNL\Documents\GitHub\RC2_Transformations\output


In [15]:
node_table = get_node_table(df, OUTPUT_DIR)
print(f'Node table: {len(node_table):,} unique masses written to {OUTPUT_DIR}')
node_table.head()

Node table: 2,405,109 unique masses written to c:\Users\gara009\OneDrive - PNNL\Documents\GitHub\RC2_Transformations\output


,Mass,C,H,O,N,S,P,OC,HC,NOSC,GFE,Class,MolecularFormula,El_comp
0,202.970305,4,4,2,4.0,2.0,0.0,0.500000,1.000000,4.000000,-53.700000,N4 S2 O2,C4H4O2S2N4,CHOSN
1,238.963145,13,4,1,0.0,2.0,0.0,0.076923,0.307692,0.153846,55.915385,S2 O1,C13H4O1S2,CHOS
2,247.097696,14,16,4,0.0,0.0,0.0,0.285714,1.142857,-0.571429,76.585714,O4,C14H16O4,CHO
3,252.978797,14,6,1,0.0,2.0,0.0,0.071429,0.428571,0.000000,60.300000,S2 O1,C14H6O1S2,CHOS
4,255.233050,16,32,2,0.0,0.0,0.0,0.125000,2.000000,-1.750000,110.175000,O2,C16H32O2,CHO


In [16]:
summary_path = os.path.join(OUTPUT_DIR, 'Transformations_summary_counts.csv')
summary = pd.read_csv(summary_path)

print(f'Total rows in summary  : {len(summary)}')
print(f'Samples                : {summary.SampleID.nunique()}')
print(f'Transformation types   : {summary.Transformation.nunique()}')

pivot = (
    summary
    .pivot_table(
        index=['Group', 'Transformation'],
        columns='SampleID',
        values='Counts',
        aggfunc='sum',
    )
    .fillna(0)
    .astype(int)
)
pivot.sort_values(by=pivot.columns.tolist(), ascending=False).head(20)

Total rows in summary  : 458500
Samples                : 575
Transformation types   : 1


,SampleID,RC2_0001_ICR-1_p08,RC2_0001_ICR-2_p08,RC2_0001_ICR-3_p08,RC2_0002_ICR-1_p08,RC2_0002_ICR-2_p08,RC2_0002_ICR-3_p08,RC2_0003_ICR-1_p08,RC2_0003_ICR-2_p08,RC2_0003_ICR-3_p08,RC2_0004_ICR-1_p08,...,RC2_0208_ICR-3_p08,RC2_0209_ICR-1_p08,RC2_0209_ICR-2_p08,RC2_0209_ICR-3_p08,RC2_0210_ICR-1_p08,RC2_0210_ICR-2_p08,RC2_0210_ICR-3_p08,RC2_0211_ICR-1_p08,RC2_0211_ICR-2_p08,RC2_0211_ICR-3_p08
Group,Transformation,,,,,,,,,,,,,,,,,,,,,
Other,random,28182,28969,27290,28475,25306,24712,25648,27178,22630,26588,...,24387,22159,23956,22884,23373,22648,25181,23240,24348,25157
